# Toggle Switch

This notebook builds a bistable toggle switch model with BioCRNpyler, exports it to SBML, and loads the SBML model into AutoReduce using the refactored public API.


In [1]:
from pathlib import Path
import numpy as np
from autoreduce import load_sbml, solve_ode


In [2]:
from biocrnpyler import Species
from biocrnpyler.components import DNAassembly, RegulatedPromoter
from biocrnpyler.mechanisms.global_mechanisms import Dilution
from biocrnpyler.mixtures import SimpleTxTlExtract

parameter_file = Path("models/design1_parameters.tsv")
sbml_file = Path("models/design1_regulated_promoter.xml")


## Build bistable toggle switch model using BioCRNpyler

The design has two regulated promoters. Each promoter has one activating regulator and one repressing regulator, and the resulting CRN is exported as SBML for AutoReduce.


In [8]:
A1 = Species("A1", material_type="protein")
A2 = Species("A2", material_type="protein")
R1 = Species("R1", material_type="protein")
R2 = Species("R2", material_type="protein")

p1 = RegulatedPromoter(
    name="p1",
    regulators=[A1, R2],
    leak=True,
)

p2 = RegulatedPromoter(
    name="p2",
    regulators=[A2, R1],
    leak=True,
)

D1 = DNAassembly(
    name="D1",
    promoter=p1,
    transcript="m1",
    rbs="utr1",
    protein=[A1, R1],
)

D2 = DNAassembly(
    name="D2",
    promoter=p2,
    transcript="m2",
    rbs="utr1",
    protein=[A2, R2],
)

protein_degradation = Dilution(
    name="protein_degradation",
    filter_dict={"protein": True, "complex": False},
    default_on=False,
)

mixture = SimpleTxTlExtract(
    name="Design1_RegulatedPromoter",
    components=[D1, D2],
    parameter_file=str(parameter_file),
    overwrite_parameters=True,
    global_mechanisms={
        "protein_degradation": protein_degradation,
    },
)

initial_conditions = {
    D1.dna: 1.0,
    D2.dna: 1.0,
}

crn = mixture.compile_crn(initial_concentration_dict=initial_conditions)

print(crn.pretty_print(show_rates=True, show_keys=True))

crn.write_sbml_file(str(sbml_file))
print(f"SBML written to {sbml_file}")


Species(N = 12) = {
    dna[D2] (@ 1.0),  
    dna[D1] (@ 1.0),  
    rna[m2] (@ 0),  
    rna[m1] (@ 0),  
    complex[dna[D2]:protein[R1]] (@ 0),  
    complex[dna[D2]:protein[A2]] (@ 0),  
    complex[dna[D1]:protein[R2]] (@ 0),  
    complex[dna[D1]:protein[A1]] (@ 0),  
    protein[R2] (@ 0),  
    protein[R1] (@ 0),  
    protein[A2] (@ 0),  
    protein[A1] (@ 0),  
}

Reactions (18) = [
0. dna[D1] --> dna[D1]+rna[m1]
 Kf=k_forward * dna_D1
  k_forward=0.25
  found_key=(mech=transcription, partid=p1_leak, name=ktx).
  search_key=(mech=simple_transcription, partid=['p1_leak', None], name=ktx).

1. protein[A1]+dna[D1] <--> complex[dna[D1]:protein[A1]]
 Kf=k_forward * protein_A1 * dna_D1
 Kr=k_reverse * complex_dna_D1_protein_A1_
  k_forward=1.0
  found_key=(mech=one_step_cooperative_binding, partid=p1_A1, name=kb).
  search_key=(mech=one_step_cooperative_binding, partid=['p1_A1', 'dna_protein', None], name=kb).
  k_reverse=1.0
  found_key=(mech=one_step_cooperative_binding, partid

c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\core\parameter.py:1575: UserWarning: parameter file contains no unit column! Please add a column named ['unit', 'units'].
  warn(
c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: species complex_dna_D1_protein_R2_ has multiple attributes (or material type) which conflict with global mechanism filter {repr(self)}. Using default value False.
  warn(
c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: species complex_dna_D1_protein_A1_ has multiple attributes (or material type) which conflict with global mechanism filter {repr(self)}. Using default value False.
  warn(
c:\Users\ayush\anaconda3\envs\autoreduce\Lib\site-packages\biocrnpyler\mechanisms\global_mechanisms.py:204: UserWarning: species complex_dna_D2_protein_R1_ has multiple attributes (or material type) which conflict w

### Verify the CRN

In [7]:
def srepr(species):
    return repr(species).replace(" ", "")

def has_species(species, text):
    return text in srepr(species)

def reaction_matches(reaction, input_tests, output_tests):
    inputs = list(reaction.inputs)
    outputs = list(reaction.outputs)

    return (
        all(any(test(s) for s in inputs) for test in input_tests)
        and all(any(test(s) for s in outputs) for test in output_tests)
    )

def find_reaction(input_tests, output_tests):
    return [
        r for r in crn.reactions
        if reaction_matches(r, input_tests, output_tests)
    ]

def dna(name):
    return lambda s: has_species(s, f"dna[{name}]")

def rna(name):
    return lambda s: has_species(s, f"rna[{name}]")

def prot(name):
    return lambda s: has_species(s, f"protein[{name}]")

def complex_with(*tokens):
    return lambda s: "complex[" in srepr(s) and all(tok in srepr(s) for tok in tokens)

checks = {
    "basal_tx_D1": find_reaction(
        [dna("D1")],
        [dna("D1"), rna("m1")],
    ),
    "basal_tx_D2": find_reaction(
        [dna("D2")],
        [dna("D2"), rna("m2")],
    ),
    "A1_binds_D1": find_reaction(
        [dna("D1"), prot("A1")],
        [complex_with("dna[D1]", "protein[A1]")],
    ),
    "R2_binds_D1": find_reaction(
        [dna("D1"), prot("R2")],
        [complex_with("dna[D1]", "protein[R2]")],
    ),
    "A2_binds_D2": find_reaction(
        [dna("D2"), prot("A2")],
        [complex_with("dna[D2]", "protein[A2]")],
    ),
    "R1_binds_D2": find_reaction(
        [dna("D2"), prot("R1")],
        [complex_with("dna[D2]", "protein[R1]")],
    ),
    "activated_tx_D1": find_reaction(
        [complex_with("dna[D1]", "protein[A1]")],
        [complex_with("dna[D1]", "protein[A1]"), rna("m1")],
    ),
    "activated_tx_D2": find_reaction(
        [complex_with("dna[D2]", "protein[A2]")],
        [complex_with("dna[D2]", "protein[A2]"), rna("m2")],
    ),
    "translation_m1_A1": find_reaction(
        [rna("m1")],
        [rna("m1"), prot("A1")],
    ),
    "translation_m1_R1": find_reaction(
        [rna("m1")],
        [rna("m1"), prot("R1")],
    ),
    "translation_m2_A2": find_reaction(
        [rna("m2")],
        [rna("m2"), prot("A2")],
    ),
    "translation_m2_R2": find_reaction(
        [rna("m2")],
        [rna("m2"), prot("R2")],
    ),
    "m1_degradation": find_reaction(
        [rna("m1")],
        [],
    ),
    "m2_degradation": find_reaction(
        [rna("m2")],
        [],
    ),
    "A1_degradation": find_reaction(
        [prot("A1")],
        [],
    ),
    "A2_degradation": find_reaction(
        [prot("A2")],
        [],
    ),
    "R1_degradation": find_reaction(
        [prot("R1")],
        [],
    ),
    "R2_degradation": find_reaction(
        [prot("R2")],
        [],
    ),
}

missing = [name for name, matches in checks.items() if len(matches) == 0]

if missing:
    print("Missing expected reaction classes:")
    for name in missing:
        print("  -", name)
else:
    print("All expected Design 1 reaction classes are present.")

for name, matches in checks.items():
    print(f"{name}: {len(matches)} match(es)")

Missing expected reaction classes:
  - basal_tx_D1
  - basal_tx_D2
  - A1_binds_D1
  - R2_binds_D1
  - A2_binds_D2
  - R1_binds_D2
  - activated_tx_D1
  - activated_tx_D2
  - translation_m1_A1
  - translation_m1_R1
  - translation_m2_A2
  - translation_m2_R2
  - m1_degradation
  - m2_degradation
  - A1_degradation
  - A2_degradation
  - R1_degradation
  - R2_degradation
basal_tx_D1: 0 match(es)
basal_tx_D2: 0 match(es)
A1_binds_D1: 0 match(es)
R2_binds_D1: 0 match(es)
A2_binds_D2: 0 match(es)
R1_binds_D2: 0 match(es)
activated_tx_D1: 0 match(es)
activated_tx_D2: 0 match(es)
translation_m1_A1: 0 match(es)
translation_m1_R1: 0 match(es)
translation_m2_A2: 0 match(es)
translation_m2_R2: 0 match(es)
m1_degradation: 0 match(es)
m2_degradation: 0 match(es)
A1_degradation: 0 match(es)
A2_degradation: 0 match(es)
R1_degradation: 0 match(es)
R2_degradation: 0 match(es)


## Load the SBML model into AutoReduce

`load_sbml` is available from the top-level `autoreduce` package after the API refactor. It returns an AutoReduce reducible system built from the BioCRNpyler-generated SBML file.


In [4]:
system = load_sbml(str(sbml_file))
len(system.x), len(system.params)

(12, 20)

## Inspect the imported system

The state names come from the SBML species IDs generated by BioCRNpyler. Inspecting them makes it easier to choose outputs or reduction assumptions in later cells.


In [5]:
for index, state in enumerate(system.x):
    print(index, state)

0 dna_D1
1 rna_m1
2 protein_A1
3 protein_R1
4 complex_dna_D1_protein_A1_
5 protein_R2
6 complex_dna_D1_protein_R2_
7 dna_D2
8 rna_m2
9 protein_A2
10 complex_dna_D2_protein_A2_
11 complex_dna_D2_protein_R1_


## Solve the imported model

The direct `solve_ode` function can solve the imported AutoReduce system without constructing an intermediate ODE solver object.


In [6]:
timepoints = np.linspace(0.0, 10.0, 101)
solution = solve_ode(system, timepoints)

solution.shape


(101, 12)